#### Pydantic 
- Pydantic is a python library used for :
    * Data Validation
    * Data Parsing
    * Type checking
    * Data Conversion
- it is a smart python class that checks weather the data is correct.

#### Why use Pydantic?
- Powered by type hints — with Pydantic, schema validation and serialization are controlled by type annotations; less to learn, less code to write, and integration with your IDE and static analysis tools.
- Speed — Pydantic’s core validation logic is written in Rust. As a result, Pydantic is among the fastest data validation libraries for Python.
- JSON Schema — Pydantic models can emit JSON Schema, allowing for easy integration with other tools
- Strict and Lax mode — Pydantic can run in either strict mode (where data is not converted) or lax mode where Pydantic tries to coerce data to the correct type where appropriate
- Dataclasses, TypedDicts and more — Pydantic supports validation of many standard library types including dataclass and TypedDict
- Customisation — Pydantic allows custom validators and serializers to alter how data is processed in many powerful ways
- Ecosystem — around 8,000 packages on PyPI use Pydantic, including massively popular libraries like FastAPI, huggingface, Django Ninja, SQLModel, & LangChain. 
- Battle tested — Pydantic is downloaded over 550M times/month and is used by all FAANG companies and 20 of the 25 largest companies on NASDAQ. If you’re trying to do something with Pydantic, someone else has probably already done it.

### Why do we need Pydantic?

In [4]:
# Lets take normal python calss
class Employee:
    def __init__(self,name,salary):
        self.name = name
        self.salary = salary

In [ ]:
# create an object
emp = Employee("Subbu",5000)
# evrything above is works in classes 
# print(vars(emp))

# but what if someone does the below 
emp = Employee(123, "abc")
print(vars(emp))
# so here python still accepts it , which make no sense. Normal classes dont automatically validate data.


{'name': 123, 'salary': 'abc'}


In [28]:
# Pydantic solves this above problem 
from pydantic import BaseModel

class Employee(BaseModel):
    name: str
    salary: int

# create an object 
emp = Employee(
    name = "subbu",
    salary = 5000
)
print(emp)

# here if we notice no constructor, no self, no init. Pydantic creates Everything Automatically.

name='subbu' salary=5000


#### Understanding Base Model

In [30]:
# normal class
class Employee():
    pass

# Pydantic class
class Employee(BaseModel):
    pass

By inherting from Basemodel, the class gets:
- validation
- type checking
- JSON Conversion
- Serialization : converting a structured BaseModel instance into a less structured format like, python dictionary or json-encoded string model_dump() and model_dump_json()
- Deserilization

##### How pydantic Reads Fields

In [31]:
class Employee(BaseModel):
    name: str
    salary: int

# here the below are fields 
name:str
salary:int 

# pydantic understands 
# 1. Name must be a string 
# 2. Salary must be a Integer 

##### Automatic Type validation

In [ ]:
emp = Employee(
    name="Subbu",
    salary=50000
)
print(emp)

# what if we pass the salary as string will see the validation error.
emp = Employee(
    name="Subbu",
    salary="hello"  # so here pydantic catches the problem immediately.
) 

name='Subbu' salary=50000


ValidationError: 1 validation error for Employee
salary
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='hello', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

#### Pydantic vs Normal Class

In [36]:
# here we need to write the lots of code
class Employee:

    def __init__(self, name, salary):

        if not isinstance(name, str):
            raise ValueError()

        if not isinstance(salary, int):
            raise ValueError()

        self.name = name
        self.salary = salary

# but in pydantic (only two lines.)
class Employee(BaseModel):
    name: str
    salary: int

##### Defualt Values 

In [ ]:
# defualt value automatically used 
from pydantic import BaseModel

class Employee(BaseModel):
    name: str
    salary: int = 50000

emp1 = Employee(name="subramani")
print(emp1) # the defualt value will take automatically

name='subramani' salary=50000


#### Optional Fields.

In [42]:
from typing import Optional
from pydantic import BaseModel

class Employee(BaseModel):
    name: str
    salary: int
    department: Optional[str] = None

# valid 
emp = Employee(
    name="Subbu",
    salary=50000
)
print(emp)

name='Subbu' salary=50000 department=None


#### List Fields

In [43]:
from pydantic import BaseModel

class Student(BaseModel):
    name: str
    subjects: list[str]

In [45]:
student = Student(
    name="Subbu",
    subjects=["Math","Physics"]
)
print(student)

name='Subbu' subjects=['Math', 'Physics']


#### Dictinary Fields 

In [46]:
from pydantic import BaseModel

class Employee(BaseModel):
    name: str
    skills: dict

In [49]:
# example 
emp = Employee(
    name="Subbu",
    skills={
        "Python":"Advanced",
        "SQL":"Intermediate"
    }
)
print(emp)

name='Subbu' skills={'Python': 'Advanced', 'SQL': 'Intermediate'}


#### Nested Models

In [51]:
## Adress model
from pydantic import BaseModel

class Address(BaseModel):
    city: str
    state: str

# inthis only 

class Employee(BaseModel):
    name: str
    address: Address

In [55]:
# creating the nested models.
emp = Employee(
    name="Subbu",
    address={
        "city":"Bangalore",
        "state":"Karnataka"
    }
)
print(emp) # here pydantic automatically converts nested dictionaries into objects

name='Subbu' address=Address(city='Bangalore', state='Karnataka')


#### Validation Erros

In [57]:
# this is why in fastapi and Langchain loves the pydantic models.
Employee(
    name="Subbu",
    salary="abc"
)

ValidationError: 1 validation error for Employee
address
  Field required [type=missing, input_value={'name': 'Subbu', 'salary': 'abc'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

#### Field Constraints

In [ ]:
from pydantic import BaseModel, Field

class Employee(BaseModel):
    name: str
    salary: int = Field(gt=0) # here meaning salary should be greater than 0

In [59]:
# More fields constraints 
from pydantic import BaseModel, Field

class Employee(BaseModel):

    name: str = Field(
        min_length=3,
        max_length=20
    )

#### Model Dump
- Convert Object to Dictionary.
- this is called serialization

In [63]:
emp = Employee(
    name="Subbu",
    salary=50000
)

print(emp.model_dump())

# model dump json
print(emp.model_dump_json())

{'name': 'Subbu'}
{"name":"Subbu"}


#### Model Validates
- suppose data comes from API

In [64]:
data = {
    "name":"Subbu",
    "salary":50000
}

In [ ]:
# validate : this is heavily used in real world application
emp = Employee.model_validate(data)
print(emp)

name='Subbu'
